**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/peyrache-2015"], check=True)
    os.chdir("/content/encore/entries/peyrache-2015")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Peyrache et al. (2015), Fig. 5d,e: optimal readout windows of the ADn to PoS GLM, from DANDI:000056

**Paper.** Peyrache A, Lacroix MM, Petersen PC, Buzsáki G (2015). Internally organized mechanisms of the head direction sense. *Nature Neuroscience* 18, 569-575. doi:10.1038/nn.3968.

**Dataset.** DANDI:000056, version 0.261106.1949, streamed from the archive (the version is discussed in `reproduce_peyrache_2015.ipynb`).

**Claim.** fig5e-readout-windows. It was recorded as blocked (`too_large`) and verified so in round 1; the round-2 verifier found that the printed timing did not support the block. It is run here, and the third verification round agreed with the outcome it gives.

**Method.** The GLM of `glm_peyrache_2015.ipynb` (`hd.glm_session`; Methods, Information measures), for PoS HD targets (hd-a or hd-b) of ADn HD cell assemblies only, the direction the claim is about, with the dense grid `hd.GLM_WINDOWS_DENSE`: Gaussian readout windows with s.d. from 20 to 251 ms at 20 per decade, the grid the criterion asks for near 50 to 200 ms, which covers both readings (a, the window is the s.d.; b, it is the full width at half maximum, 2.355 s.d.), and 10, 500 and 1000 ms outside it. As in that notebook, at most the first 20 minutes of each state are used. Targets are kept when positively predicted at one window or more (fig5d-adn-predicts-pos-reliably), and each kept target's best window is the one with the highest cross-validated information.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import time, resource
import numpy as np
import pandas as pd
from scipy import stats
import hd
pd.set_option("display.width", 200)
print("dandiset", hd.DANDISET, "version", hd.VERSION, "| readout window s.d. (ms):", np.round(hd.GLM_WINDOWS_DENSE * 1000, 1).tolist(),
      "| at most", hd.GLM_MAX_S, "s of each state")

In [ ]:
# The dense-window GLM on every session of the mice whose files hold units
# labelled ADn and PoS, PoS targets only.
t0 = time.time()
G = hd.run_glm(["Mouse20", "Mouse24", "Mouse25", "Mouse28"], workers=4, dense=True)
print(f"analysed {len(G)} sessions in {(time.time() - t0) / 60:.1f} min; "
      f"peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
GLM = pd.DataFrame([g for o in G for g in o["glm"]])

## Data check

In [ ]:
print("seconds per session:", {o["meta"]["path"].split("_ses-")[1].split("_")[0]: round(o["meta"]["seconds"]) for o in G})
print("sessions with a GLM (HD cells in both structures):", GLM.session.nunique(), "by mouse:", GLM.groupby("subject").session.nunique().to_dict())
print("target regions:", GLM.target_region.unique().tolist(), "| windows per target:", GLM.groupby(["session", "state", "target"]).size().unique().tolist())
print("PoS targets per state:", GLM.groupby("state").target.size().div(len(hd.GLM_WINDOWS_DENSE)).astype(int).to_dict())
lw = np.log10(hd.GLM_WINDOWS_DENSE)
print("windows per decade between 50 and 200 ms (s.d.):", round(1 / np.median(np.diff(lw[(hd.GLM_WINDOWS_DENSE >= 0.05) & (hd.GLM_WINDOWS_DENSE <= 0.2)])), 1),
      "| as full width (x 2.355), windows inside 50 to 200 ms:", int(((hd.GLM_WINDOWS_DENSE * 2.3548 >= 0.05) & (hd.GLM_WINDOWS_DENSE * 2.3548 <= 0.2)).sum()))
print("information range (bits/spike):", round(GLM["info"].min(), 3), "to", round(GLM["info"].max(), 3), "| non-finite:", int((~np.isfinite(GLM["info"])).sum()))

## fig5e-readout-windows

> The optimal readout windows of the ADn to PoS prediction are similar in waking and REM, median 123 ms and 133 ms, and shorter in SWS, median 79 ms.

Readings (criterion): (a) the window is the Gaussian's s.d., (b) its full width at half maximum. Reproduced when all three medians are within tolerance under one reading (123 plus or minus 12.3, 133 plus or minus 13.3 and 79 plus or minus 7.9 ms); partially reproduced when the SWS median is below both others and some medians are outside. The kept-target selection and hd-a targets are shown as variants, with the medians by mouse and the count of best windows at the ends of the grid.

In [ ]:
best = GLM.loc[GLM.groupby(["session", "state", "target"])["info"].idxmax()]
best = best.assign(kept=best["info"] > 0)
EXP = {"wake": (123, 12.3), "REM": (133, 13.3), "SWS": (79, 7.9)}
rows = []
for tg in ("hd-a or hd-b", "hd-a"):
    for sel in ("kept", "all targets"):
        b0 = best if tg == "hd-a or hd-b" else best[best.hd_a.astype(bool)]
        if sel == "kept":
            b0 = b0[b0.kept]
        for rd, f in (("a s.d.", 1.0), ("b FWHM", 2.3548)):
            m = {st: 1000 * f * b0[b0.state == st].window.median() for st in ("wake", "REM", "SWS")}
            rows.append(dict(targets=tg, selection=sel, reading=rd, n_wake=int((b0.state == "wake").sum()),
                             n_REM=int((b0.state == "REM").sum()), n_SWS=int((b0.state == "SWS").sum()),
                             wake_ms=m["wake"], REM_ms=m["REM"], SWS_ms=m["SWS"],
                             within=sum(abs(m[st] - EXP[st][0]) <= EXP[st][1] for st in EXP),
                             SWS_shortest=m["SWS"] < min(m["wake"], m["REM"])))
F5W = pd.DataFrame(rows)
print(F5W.to_string(index=False, float_format=lambda x: f"{x:.1f}"))
print("expected medians: wake 123 (+-12.3), REM 133 (+-13.3), SWS 79 (+-7.9) ms")
k = best[best.kept]
print("kept targets (hd-a or hd-b), median best window s.d. (ms) by mouse:")
print(k.groupby(["subject", "state"]).window.median().mul(1000).round(1).unstack().to_string())
print("best window s.d. (ms), quartiles per state:", {st: np.round(1000 * k[k.state == st].window.quantile([0.25, 0.5, 0.75]).values, 1).tolist() for st in ("wake", "REM", "SWS")})
print("best windows at an end of the grid (10 or 1000 ms):", k.groupby("state").window.apply(lambda w: int(((w <= 0.0101) | (w >= 0.999)).sum())).to_dict(),
      "| outside the dense part (10, 500 or 1000 ms):", k.groupby("state").window.apply(lambda w: int(((w < 0.019) | (w > 0.26)).sum())).to_dict())
for a_, b_ in (("SWS", "wake"), ("SWS", "REM"), ("wake", "REM")):
    u = stats.mannwhitneyu(k[k.state == a_].window, k[k.state == b_].window, alternative="two-sided")
    print(f"Mann-Whitney best window {a_} against {b_}: p {u.pvalue:.3g}")

**Result: partially reproduced.** With the windows read as the Gaussian's s.d. (a) and the kept PoS targets (HD cells under either reading), the median best window is 168 ms in waking, 178 ms in REM and 63 ms in SWS, against 123, 133 and 79 ms: none is within tolerance, and the SWS median is below both others. Read as the full width (b), the medians are 396, 419 and 149 ms, further off. With hd-a targets (a), they are 200, 224 and 67 ms. Every variant has the SWS median the shortest, and the SWS windows are shorter than the waking and REM ones (Mann-Whitney p = 1.6e-18 and 6.9e-12). Waking and REM are similar, as the claim says (Mann-Whitney p = 0.96), but both are longer than the paper's. The best window lies at an end of the grid (10 or 1000 ms) for 10 waking, 22 REM and 6 SWS targets, so the REM median in particular is loose; its interquartile range is 89 to 500 ms. So the claim is partially reproduced under the criterion. The third verification round agreed with this outcome.

## Summary

| claim | outcome |
|---|---|
| fig5e-readout-windows | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** HD cells as assembly and targets (readings g-b, g-c); PoS targets of ADn assemblies only; 26 readout windows, 23 of them at 20 per decade from 20 to 251 ms s.d.; at most the first 20 minutes of each state; Newton's method in place of a derivative-free search.